In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from torch.distributions.normal import Normal
from scipy.interpolate import griddata
%matplotlib inline

# ================= 路径配置 =================
# 设置保存结果的根目录（根据你的需求修改）
RESULTS_ROOT = "./generated/pinn_results"  

# 为此次运行创建带有时间戳的子文件夹
timestamp = time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = f"run_pi100_K50_{timestamp}"
SAVE_DIR = os.path.join(RESULTS_ROOT, RUN_DIR)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Results will be saved to: {SAVE_DIR}")

# ================= 参数设置 =================
pi = 100.0   # 参数 pi
beta = 0.5   # 参数 beta
alpha = 1.2  # 参数 alpha
r = 0.05     # 无风险利率
T = 4.0      # 资产价格相关参数
tau = 3.0    # 期权到期时间
K = 50.0     # 行权价

# ================= 随机种子 =================
SEED = 101
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normal = Normal(0, 1)  # 标准正态分布

# ================= 神经网络定义 (修复维度问题) =================
class CarbonOptionNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, A, t):
        # 确保输入是二维张量 (batch_size, feature_size)
        # 修复维度问题
        if A.dim() == 1:
            A = A.view(-1, 1)
        if t.dim() == 1:
            t = t.view(-1, 1)
        x = torch.cat([A, t], dim=1)  # 连接A和t特征
        return self.net(x)

# ================= 工具函数 (修复维度问题) =================
def derivative(output, input_var, order=1):
    # 如果输入是1D，转换为2D
    if output.dim() == 1:
        output = output.view(-1, 1)
    
    grad = output
    for _ in range(order):
        grad = torch.autograd.grad(
            outputs=grad,
            inputs=input_var,
            grad_outputs=torch.ones_like(grad),
            create_graph=True,
            retain_graph=True,
        )[0]
        
        # 如果grad变为None（有时PyTorch会这样），使用零张量
        if grad is None:
            grad = torch.zeros_like(input_var)
    
    return grad

def dfdt(model, A, t, order=1):
    t_requires_grad = t.requires_grad
    if not t_requires_grad:
        t.requires_grad_(True)
    
    C = model(A, t)
    dC_dt = derivative(C, t, order)
    
    # 恢复原始梯度设置
    if not t_requires_grad:
        t.requires_grad_(False)
    
    return dC_dt

def dfdx(model, A, t, order=1):
    A_requires_grad = A.requires_grad
    if not A_requires_grad:
        A.requires_grad_(True)
    
    C = model(A, t)
    dC_dA = derivative(C, A, order)
    
    # 恢复原始梯度设置
    if not A_requires_grad:
        A.requires_grad_(False)
    
    return dC_dA

def final_condition(A, K):
    # 确保A是2D张量
    if A.dim() == 1:
        A = A.view(-1, 1)
    return torch.maximum(A - K, torch.tensor(0.0, device=A.device)).reshape(-1, 1)

# ================= PDE损失函数 =================
def pde_loss(model, A, t, pi, beta, alpha, r, T, tau):
    # 确保输入是二维张量
    if A.dim() == 1:
        A = A.view(-1, 1).requires_grad_(True)
    if t.dim() == 1:
        t = t.view(-1, 1).requires_grad_(True)
    
    # 计算导数
    dC_dt = dfdt(model, A, t, order=1)
    d2C_dA2 = dfdx(model, A, t, order=2)
    
    # 计算相关项
    z_t = beta * (T - t).pow(-alpha)
    
    # 处理零除和小数值
    A_pi_ratio = A / pi
    A_pi_ratio = A_pi_ratio.clamp(min=1e-6, max=1-1e-6)
    
    Phi_inv = normal.icdf(A_pi_ratio)
    Phi_prime = torch.exp(-0.5 * Phi_inv.pow(2)) / torch.sqrt(torch.tensor(2.0 * torch.pi))
    
    # 非线性项
    nonlinear_term = 0.5 * d2C_dA2 * (pi * Phi_prime * torch.sqrt(z_t)).pow(2)
    
    # PDE残差
    pde_residual = dC_dt + nonlinear_term - r * model(A, t)
    
    return pde_residual.pow(2).mean()

# ================= 总损失函数 (修复维度问题) =================
def compute_loss(model, A, t, A_domain, t_domain, pi, beta, alpha, r, T, tau, K):
    # PDE损失
    pde_loss_val = pde_loss(model, A, t, pi, beta, alpha, r, T, tau)
    
    # 确保边界张量是2D
    if t_domain.dim() == 1:
        t_domain = t_domain.view(-1, 1)
    if A_domain.dim() == 1:
        A_domain = A_domain.view(-1, 1)
    # 边界A=0条件（保持不变）
    boundary_A0 = torch.ones_like(t_domain) * A_domain[0]
    boundary_loss_A0 = model(boundary_A0, t_domain)
    
    # 修正边界A=max条件 - 使用正确的到期时间τ
    boundary_A1 = torch.ones_like(t_domain) * A_domain[-1]
    
    # 关键修正：使用τ而不是T来计算折现！
    discount = torch.exp(-r * (tau - t_domain))  # τ是期权到期时间
    
    boundary_loss_A1 = model(boundary_A1, t_domain) - (A_domain[-1] - K * discount)
    
    # 终端条件 - 增加权重
    final_weight = 50.0  # 增加终端条件权重
    # 终端条件 (t=tau)
    f_final = final_condition(A_domain, K)
    t_final = torch.ones_like(A_domain) * tau
    
    # 如果需要，为A_domain设置梯度
    if not A_domain.requires_grad:
        A_domain.requires_grad_(True)
    
    final_loss = model(A_domain, t_final) - f_final
    # 总损失
    loss_val = (
        10 * pde_loss_val +
        boundary_loss_A0.pow(2).mean() +
        # 100.0 * boundary_loss_A1.pow(2).mean() +  # 应用权重
        final_weight * final_loss.pow(2).mean()  # 应用权重
    )
    
    return loss_val
    # 边界A=0条件
    # boundary_A0 = torch.ones_like(t_domain) * A_domain[0]
    # boundary_loss_A0 = model(boundary_A0, t_domain)
    
    # # 边界A=max条件
    # boundary_A1 = torch.ones_like(t_domain) * A_domain[-1]
    # discount = torch.exp(-r * (T - t_domain))
    # boundary_loss_A1 = model(boundary_A1, t_domain) - (A_domain[-1] - K * discount)
    
    
    
    # # 总损失
    # loss_val = (
    #     pde_loss_val +
    #     boundary_loss_A0.pow(2).mean() +
    #     100 * boundary_loss_A1.pow(2).mean() +
    #     2e1 * final_loss.pow(2).mean()
    # )
    
    # return loss_val

# ================= 数据准备 (修复维度问题) =================
A_min, A_max = 0.0, 100.0
t_min, t_max = 0.0, tau
n_points = 150  # 减少点数加速训练

# 创建带梯度的输入点
A_domain_tensor = torch.linspace(A_min, A_max, n_points, requires_grad=True, device=device)
t_domain_tensor = torch.linspace(t_min, t_max, n_points, requires_grad=True, device=device)

# 创建训练网格
A_grid, t_grid = torch.meshgrid(A_domain_tensor, t_domain_tensor, indexing='ij')
A_flat = A_grid.reshape(-1, 1)
t_flat = t_grid.reshape(-1, 1)

# ================= 模型训练 =================
model = CarbonOptionNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss = []
print("Starting training...")
start_time = time.time()

# 简化训练循环
for epoch in range(15001):
    optimizer.zero_grad()
    
    # 使用更可靠的数据传递方式
    loss = compute_loss(
        model, 
        A_flat.detach().requires_grad_(True), 
        t_flat.detach().requires_grad_(True),
        A_domain_tensor,
        t_domain_tensor,
        pi, beta, alpha, r, T, tau, K
    )
    
    loss.backward()
    optimizer.step()
    
    train_loss.append(loss.item())
    if epoch % 500 == 0:
        print(f"Epoch {epoch:4d}: Loss = {loss.item():.8f}")

training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

# ================= 保存训练损失 =================
loss_data = np.array(train_loss)
loss_path = os.path.join(SAVE_DIR, "training_loss.npy")
np.save(loss_path, loss_data)

# ================= 结果预测 =================
with torch.no_grad():
    # 确保输入是二维的
    predictions = model(A_flat, t_flat)
    predictions = predictions.cpu().numpy()

# 保存预测结果
A_vals = A_flat.cpu().detach().numpy()
t_vals = t_flat.cpu().detach().numpy()

results = np.hstack((A_vals, t_vals, predictions))
results_path = os.path.join(SAVE_DIR, "pinn_results.npy")
np.save(results_path, results)

# 保存模型
model_path = os.path.join(SAVE_DIR, "carbon_option_model.pth")
torch.save(model.state_dict(), model_path)


# ================= 保存参数信息 =================
params_info = f"""
Model Parameters:
----------------
Pi = {pi}
Beta = {beta}
Alpha = {alpha}
Risk-free rate (r) = {r}
T = {T}
Tau = {tau}
Strike price (K) = {K}

Training Parameters:
-------------------
Points per dimension = {n_points}
Epochs = {5001}
Learning rate = {1e-3}
Training time = {training_time:.2f} seconds
"""

params_path = os.path.join(SAVE_DIR, "simulation_parameters.txt")
with open(params_path, "w") as f:
    f.write(params_info)

print(f"All results saved to: {SAVE_DIR}")